In [ ]:
import pandas as pd
import numpy as np
import itertools
from plotnine import (
    ggplot,
    aes,
    geom_density,
    facet_wrap,
    theme_bw,
    labs,
    theme,
    geom_vline,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from funs import (
    dataPreparation,
    createMetaDictionary,
    createDictionary,
    dictionaryModel,
    evaluateModel,
)

# Data preparation

In [ ]:
data = dataPreparation(
    all_trxns_path="../data/all_trxns.csv", exchange_rates_path="../data/exchange_rates.csv"
)

X = data
y = data["fraud_flag"]

# Stratified split preserves the ~1.7% positive rate in both folds. A
# chronological split would risk leaving the test set with too few (or zero)
# frauds at this imbalance level; stratification is the safer default here.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fraud probability dictionaries model

This approach tailored to fraud detection through statistical analysis and rule-based flagging, relying on the assumption that certain statistical patterns in the data can be indicative of fraudulent activity.

In [ ]:
dictionaries_to_get = [
    "customer_country",
    "counterparty_country",
    "type",
    "ccy",
    "customer_type",
    "weekday",
    "month",
    "quarter",
    "hour",
    "amount_eur_bucket",
]

## Check historical fraud probability for each value of each variable

In [ ]:
dictionaries = {}
for dict_name in dictionaries_to_get:
    dictionaries[dict_name] = createDictionary(
        X_train, colname=dict_name, count_filter=0
    )

## Define the limits for the historical fraud probability for each variable

In [ ]:
meta_dicts = []  # Use a list to collect dictionaries
for dict_name in dictionaries_to_get:
    meta_dict = createMetaDictionary(X_train, colname=dict_name, quantile_threshold=0.9, count_filter=0)
    meta_dicts.append(meta_dict)

In [ ]:
meta_dictionary = pd.concat(meta_dicts, ignore_index=True)

## Model
1. Join and transform the data via `dictionaryModel` (shared harness in `funs.py`)
    - Train Input Data + Dictionaries + Meta Dictionary
2. Aggregate to transaction level: `expected_fraud_probability` (sum of per-variable fraud probabilities) and the flag counts (`sd_flags`, `quantile_flags`)
3. Search the threshold space on the TRAIN set to maximize F1, then apply the SAME rule to test (no train/test rule divergence)

In [ ]:
# Build the transaction-level aggregation via the shared harness. Thresholds are
# passed as permissive dummies here (everything predicts positive) so the
# aggregation columns are populated; the actual prediction rule is fit by the
# grid search below and applied consistently to both train and test.
joined_data_aggregated = dictionaryModel(
    X_train,
    dictionaries,
    meta_dictionary,
    fraud_probability_threshold=0.0,
    sd_flags_threshold=-1,
    quantile_flags_threshold=-1,
)

In [ ]:
training_set = joined_data_aggregated

### Training Model setup

In [ ]:
grouped_data = training_set.groupby("fraud_flag").agg(
    mean_value=("expected_fraud_probability", "mean"),
    q1_value=("expected_fraud_probability", lambda x: x.quantile(0.1)),
    q25_value=("expected_fraud_probability", lambda x: x.quantile(0.25)),
    q75_value=("expected_fraud_probability", lambda x: x.quantile(0.75)),
    q9_value=("expected_fraud_probability", lambda x: x.quantile(0.9)),
    sd_flags=("sd_flags", "sum"),
    quantile_flags=("quantile_flags", "sum"),
    q_1_flags=("quantile_1_flags", "sum"),
    q_25_flags=("quantile_25_flags", "sum"),
    q_75_flags=("quantile_75_flags", "sum"),
    q_9_flags=("quantile_9_flags", "sum"),
    n=("expected_fraud_probability", "count"),
)
grouped_data["sd_mean_flags"] = grouped_data["sd_flags"] / grouped_data["n"]
grouped_data["quantile_mean_flags"] = grouped_data["quantile_flags"] / grouped_data["n"]
grouped_data["q_1_mean_flags"] = grouped_data["q_1_flags"] / grouped_data["n"]
grouped_data["q_25_mean_flags"] = grouped_data["q_25_flags"] / grouped_data["n"]
grouped_data["q_75_mean_flags"] = grouped_data["q_75_flags"] / grouped_data["n"]
grouped_data["q_9_mean_flags"] = grouped_data["q_9_flags"] / grouped_data["n"]
grouped_data = grouped_data.reset_index()

#### Training Model Analysis

In [ ]:
grouped_data

### Model Expected Fraud Probability

In [ ]:
print(
    "Overall Expected Fraud Probability: ",
    training_set["expected_fraud_probability"].mean().round(4),
)

In [ ]:
(
    ggplot(training_set, aes(x="expected_fraud_probability", fill="fraud_flag"))
    + geom_density(alpha=0.5)
    + facet_wrap("~ fraud_flag", scales="free_y")
    + theme_bw()
    + labs(
        title="Expected Fraud Probability Distribution by Fraud Flag",
        x="",
        y="",
        fill="Fraud flag",
        caption="Red vertical line is mean value for the whole dataset",
    )
    + theme(legend_position="top")
    + geom_vline(
        xintercept=training_set["expected_fraud_probability"].mean(),
        color="red",
        linetype="dashed",
        size=0.5,
    )
)

This looks promissing: the distribution of the `expected_fraud_probability` is different for fraud and non-fraud transactions, it is shifted to the right for fraud transactions.

It could indicate that model that contains the information that could help to identify the fraud. 

### Model Standard Deviation Flags

In [ ]:
print("Mean of sd_flags: ", training_set["sd_flags"].mean().round(2))

In [ ]:
(
    ggplot(training_set, aes(x="sd_flags", fill="fraud_flag"))
    + geom_density(alpha=0.5)
    + facet_wrap("~ fraud_flag", scales="free_y")
    + theme_bw()
    + labs(
        title="Number of SD Flags by Fraud Flag",
        x="",
        y="",
        fill="Fraud flag",
        caption="Red vertical line is mean value for the whole dataset",
    )
    + theme(legend_position="top", subplots_adjust={"wspace": 0.25})
    + geom_vline(
        xintercept=training_set["sd_flags"].mean(),
        color="red",
        linetype="dashed",
        size=0.5,
    )
)

This may serve as a significant indicator. It suggests that the likelihood of detecting fraud increases as more flags are identified in an observation.

### Quantile Flags

In [ ]:
print("Mean of quantile flags: ", training_set["quantile_flags"].mean().round(2))

In [ ]:
(
    ggplot(training_set, aes(x="quantile_flags", fill="fraud_flag"))
    + geom_density(alpha=0.5)
    + facet_wrap("~ fraud_flag", scales="free_y")
    + theme_bw()
    + labs(
        title="Number of Quantile Flags by Fraud Flag",
        x="",
        y="",
        fill="Fraud flag",
        caption="Red vertical line is mean value for the whole dataset",
    )
    + theme(legend_position="top")
    + geom_vline(
        xintercept=training_set["quantile_flags"].mean(),
        color="red",
        linetype="dashed",
        size=0.5,
    )
)

This may serve as a significant indicator. It suggests that the likelihood of detecting fraud increases as more flags are identified in an observation.

### Model Fraud thresholds
Principled search (no hand-tuning): a grid over the three thresholds is
evaluated on the TRAIN set; the combination maximizing F1 is selected and then
applied unchanged to the TEST set.

1. `expected_fraud_probability`
2. `sd_flags`
3. `quantile_flags`

In [ ]:
# Grid search over the threshold space, maximizing F1 on the train set.
# The score column (expected_fraud_probability) is also passed to the eval
# harness as y_score so PR-AUC / ROC-AUC are reported alongside accuracy.
fp_grid = np.arange(0.05, 0.40, 0.025)
flag_grid = [0, 1, 2, 3]

best_f1 = -1.0
best_thresholds = None
y_train_true = joined_data_aggregated["fraud_flag_transformed"].values
for fp_t, sd_t, qf_t in itertools.product(fp_grid, flag_grid, flag_grid):
    y_pred = (
        (joined_data_aggregated["expected_fraud_probability"] > fp_t)
        & (joined_data_aggregated["sd_flags"] > sd_t)
        & (joined_data_aggregated["quantile_flags"] > qf_t)
    ).astype(int)
    f1 = f1_score(y_train_true, y_pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_thresholds = (float(fp_t), int(sd_t), int(qf_t))

fp_threshold, sd_threshold, qf_threshold = best_thresholds
print(
    "Best F1=%.4f on train with thresholds "
    "(expected_fraud_probability>%.3f, sd_flags>%d, quantile_flags>%d)"
    % (best_f1, fp_threshold, sd_threshold, qf_threshold)
)

result = joined_data_aggregated.assign(
    predicted_fraud=np.where(
        (joined_data_aggregated["expected_fraud_probability"] > fp_threshold)
        & (joined_data_aggregated["sd_flags"] > sd_threshold)
        & (joined_data_aggregated["quantile_flags"] > qf_threshold),
        1,
        0,
    )
)

### Train Model Evaluation

In [ ]:
train_model = X_train[["timestamp", "customer", "counterparty", "fraud_flag"]].merge(
    result.drop(columns="fraud_flag"),
    on=["timestamp", "customer", "counterparty"],
    how="left",
)

train_model_prediction = train_model.groupby(["fraud_flag", "predicted_fraud"]).agg(
    mean_value=("expected_fraud_probability", "mean"),
    q1_value=("expected_fraud_probability", lambda x: x.quantile(0.1)),
    q25_value=("expected_fraud_probability", lambda x: x.quantile(0.25)),
    q75_value=("expected_fraud_probability", lambda x: x.quantile(0.75)),
    q9_value=("expected_fraud_probability", lambda x: x.quantile(0.9)),
    sd_flags=("sd_flags", "sum"),
    quantile_flags=("quantile_flags", "sum"),
    quantile_1_flags=("quantile_1_flags", "sum"),
    quantile_25_flags=("quantile_25_flags", "sum"),
    quantile_75_flags=("quantile_75_flags", "sum"),
    quantile_9_flags=("quantile_9_flags", "sum"),
    n=("expected_fraud_probability", "count"),
)
train_model_prediction["sd_mean_flags"] = (
    train_model_prediction["sd_flags"] / train_model_prediction["n"]
)
train_model_prediction["quantile_mean_flags"] = (
    train_model_prediction["quantile_flags"] / train_model_prediction["n"]
)

train_model_prediction["quantile_1_mean_flags"] = (
    train_model_prediction["quantile_1_flags"] / train_model_prediction["n"]
)

train_model_prediction["quantile_25_mean_flags"] = (
    train_model_prediction["quantile_25_flags"] / train_model_prediction["n"]
)

train_model_prediction["quantile_75_mean_flags"] = (
    train_model_prediction["quantile_75_flags"] / train_model_prediction["n"]
)

train_model_prediction["quantile_9_mean_flags"] = (
    train_model_prediction["quantile_9_flags"] / train_model_prediction["n"]
)
train_model_prediction = train_model_prediction.reset_index()

train_model_eval = train_model.assign(
    fraud_flag_transformed=np.where((train_model["fraud_flag"] == "Y"), 1, 0)
).dropna()

In [ ]:
train_model_prediction

In [ ]:
evaluateModel(
    train_model_eval["fraud_flag_transformed"],
    train_model_eval["predicted_fraud"],
    y_score=train_model_eval["expected_fraud_probability"],
)

### Predictive Model Definition
Apply the SAME thresholds found by the grid search on the train set to the test
set (no re-tuning, no structural divergence between train and test rules).

In [ ]:
# Apply the grid-searched thresholds to the test set via the shared harness.
# The rule is identical to the train rule: same thresholds, same three clauses.
test_model = dictionaryModel(
    X_test,
    dictionaries,
    meta_dictionary,
    fraud_probability_threshold=fp_threshold,
    sd_flags_threshold=sd_threshold,
    quantile_flags_threshold=qf_threshold,
)

### Prediction Model Evaulation

In [ ]:
evaluateModel(
    test_model["fraud_flag_transformed"],
    test_model["predicted_fraud"],
    y_score=test_model["expected_fraud_probability"],
)

In [ ]:
test_model_prediction = test_model.groupby(["fraud_flag", "predicted_fraud"]).agg(
    mean_value=("expected_fraud_probability", "mean"),
    q1_value=("expected_fraud_probability", lambda x: x.quantile(0.1)),
    q25_value=("expected_fraud_probability", lambda x: x.quantile(0.25)),
    q75_value=("expected_fraud_probability", lambda x: x.quantile(0.75)),
    q9_value=("expected_fraud_probability", lambda x: x.quantile(0.9)),
    sd_flags=("sd_flags", "sum"),
    quantile_flags=("quantile_flags", "sum"),
    quantile_1_flags=("quantile_1_flags", "sum"),
    quantile_25_flags=("quantile_25_flags", "sum"),
    quantile_75_flags=("quantile_75_flags", "sum"),
    quantile_9_flags=("quantile_9_flags", "sum"),
    n=("expected_fraud_probability", "count"),
)
test_model_prediction["sd_mean_flags"] = (
    test_model_prediction["sd_flags"] / test_model_prediction["n"]
)
test_model_prediction["quantile_mean_flags"] = (
    test_model_prediction["quantile_flags"] / test_model_prediction["n"]
)

test_model_prediction["quantile_1_mean_flags"] = (
    test_model_prediction["quantile_1_flags"] / test_model_prediction["n"]
)

test_model_prediction["quantile_25_mean_flags"] = (
    test_model_prediction["quantile_25_flags"] / test_model_prediction["n"]
)

test_model_prediction["quantile_75_mean_flags"] = (
    test_model_prediction["quantile_75_flags"] / test_model_prediction["n"]
)

test_model_prediction["quantile_9_mean_flags"] = (
    test_model_prediction["quantile_9_flags"] / test_model_prediction["n"]
)
test_model_prediction = test_model_prediction.reset_index()

test_model_prediction

### Comparative Performance with Tree-Based Models

- The model is noted for identifying more Positives compared to tree-based models.
- The overall accuracy being lower than tree-based models.

### Potential for Improvement

- Thresholds fine-tuning. 
- Integrating the model with Reinforcement Learning. This could allow the model to adaptively learn and update its parameters based on feedback, which could be particularly useful in dynamic environments like fraud detection.
- Updating dictionary values when the model makes incorrect predictions is an example of a feedback mechanism that could improve the model's performance over time.

### Combining Models for Robustness

- Dictionary-based model could be used as a Feature Engineering tool providing additional data for ML models. 